In [ ]:
from dotenv import load_dotenv
load_dotenv()
from typing_extensions import TypedDict, Annotated
from langchain.chat_models import init_chat_model
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
 ̰

In [19]:
from langchain.chat_models import init_chat_model

In [ ]:

# using the gemini 2.5 flash model, which is optimized for speed and cost-efficiency, with a temperature of 0 for deterministic responses
# this is a normal communication of the chat LLM
llm = init_chat_model(
    "google_genai:gemini-2.5-flash",
    temperature=0
)

response = llm.invoke("Who was the third Prime Minister of Saudi Arabia?")
result = response.text
print(result)


The King of Saudi Arabia also serves as the Prime Minister. Therefore, the third Prime Minister of Saudi Arabia was the third King of Saudi Arabia.

1.  **King Abdulaziz ibn Saud** (founder and first King/PM)
2.  **King Saud bin Abdulaziz Al Saud** (second King/PM)
3.  **King Faisal bin Abdulaziz Al Saud** (third King/PM)

So, the third Prime Minister of Saudi Arabia was **King Faisal bin Abdulaziz Al Saud**.


In [ ]:
# now lets make this using the stategraph

class State(TypedDict):
    # add_messages here is called the reducer function which helps in appending the new messages to to the list 
    messages : Annotated[list,add_messages]
def chatbot(state : State) -> State:
    return {"messages":llm.invoke(state["messages"])}


builder = StateGraph(State)
builder.add_node("chatbot_node", chatbot)
builder.add_edge(START, "chatbot_node")
builder.add_edge("chatbot_node", END)

graph=builder.compile()




In [25]:
message = {"role": "user", "content": "Who is the richest person in hyderabad? Print only the name"}
# message = {"role": "user", "content": "What is the latest price of MSFT stock?"}
response = graph.invoke({"messages":[message]})
response["messages"]


[HumanMessage(content='Who is the richest person in hyderabad? Print only the name', additional_kwargs={}, response_metadata={}, id='ceb2007a-257a-4fe5-9aba-b1aee81172d5'),
 AIMessage(content='Murali Divi', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d29f8-fe1d-7620-a52c-b2e25b2a3f4a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 14, 'output_tokens': 180, 'total_tokens': 194, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 176}})]

In [27]:
state = None
while True:
    in_message = input("You: ")
    if in_message.lower() in {"quit","exit"}:
        break
    if state is None:
        state: State = {
            "messages": [{"role": "user", "content": in_message}]
        }
    else:
        state["messages"].append({"role": "user", "content": in_message})

    state = graph.invoke(state)
    print("Bot:", state["messages"][-1].content)

Bot: It is not possible to identify a single "poorest person" in Hyderabad. Poverty affects many individuals, and their financial situations are private. As an AI, I do not have access to such personal information, nor would it be appropriate or ethical to disclose it.
Bot: Could you please clarify what you mean by "break"?

Are you asking me to:
*   Take a break?
*   Break down a concept or explanation?
*   Stop our current conversation?
*   Something else entirely?

I'm ready for your next instruction!
Bot: Okay, I understand. I will stop our current conversation.

If you have any other questions or need assistance in the future, feel free to start a new chat!
